In [1]:
import xarray as xr

file_path = 'Berkeley_temperature_by_region_shifted_with_ERA5_2025.nc'

try:
    ds = xr.open_dataset(file_path, engine='netcdf4')
    print(ds)
except Exception as e:
    print(f"读取文件时出错: {e}")

<xarray.Dataset> Size: 2MB
Dimensions:      (month: 12, year: 175, region_id: 237)
Coordinates:
  * month        (month) int64 96B 3 4 5 6 7 8 9 10 11 12 1 2
  * year         (year) int64 1kB 1850 1851 1852 1853 ... 2021 2022 2023 2024
  * region_id    (region_id) int64 2kB 0 1 2 3 4 5 6 ... 231 232 233 234 235 236
Data variables:
    temperature  (month, year, region_id) float32 2MB ...


In [9]:
import xarray as xr

# 打开 nc 文件
ds = xr.open_dataset('ERA5_region_t2m_1940_2024_currentyear03_to_nextyear02.nc')

# 查看变量名、坐标维度、属性
print(ds)

<xarray.Dataset> Size: 2MB
Dimensions:    (region_id: 237, year: 85, month: 12)
Coordinates:
  * region_id  (region_id) int64 2kB 0 1 2 3 4 5 6 ... 231 232 233 234 235 236
  * year       (year) int64 680B 1940 1941 1942 1943 ... 2021 2022 2023 2024
  * month      (month) int64 96B 3 4 5 6 7 8 9 10 11 12 1 2
Data variables:
    t2m        (region_id, year, month) float64 2MB ...


In [10]:
import xarray as xr
import numpy as np

# === 读取数据 ===
ds_berkeley = xr.open_dataset('Berkeley_temperature_by_region_shifted_with_ERA5_2025.nc')
ds_era5 = xr.open_dataset('ERA5_region_t2m_1940_2024_currentyear03_to_nextyear02.nc')

# 提取变量
berkeley_temp = ds_berkeley['temperature']  # (month, year, region_id) 异常值
era5_t2m = ds_era5['t2m']                  # (region_id, year, month)

# ERA5的月份顺序，确认是3-12, 1, 2
era5_months = ds_era5['month'].values
assert np.array_equal(era5_months, [3,4,5,6,7,8,9,10,11,12,1,2])

# 计算ERA5气候态 (1961-1990)
era5_clim = era5_t2m.sel(year=slice(1961, 1990)).mean(dim='year')  # (region_id, month)

# Berkeley数据结构
berkeley_months = ds_berkeley['month'].values  # (3,4,5,...12,1,2)
berkeley_years = ds_berkeley['year'].values
region_ids = ds_berkeley['region_id'].values

# 映射月信息
month_to_idx_era5 = {m: i for i, m in enumerate(era5_months)}
month_to_idx_berkeley = {m: i for i, m in enumerate(berkeley_months)}

# 复制数据，准备加气候态
temp_with_clim = berkeley_temp.copy().values  # (month, year, region_id)

# 遍历月份添加气候态
for m in berkeley_months:
    idx_month_berkeley = month_to_idx_berkeley[m]
    idx_month_era5 = month_to_idx_era5[m]

    clim_vals = era5_clim.isel(month=idx_month_era5).values  # (region_id,)

    # 默认所有年份添加
    temp_with_clim[idx_month_berkeley, :, :] += clim_vals[np.newaxis, :]

# 最后一年的1、2月不加气候态，恢复原异常值
last_year_idx = -1
months_1_2_idx = [month_to_idx_berkeley[m] for m in [1, 2]]

# 覆盖为原始异常值
temp_with_clim[months_1_2_idx, last_year_idx, :] = berkeley_temp.isel(month=months_1_2_idx, year=last_year_idx).values

# === 保存新数据 ===
ds_out = xr.Dataset(
    {
        'temperature': (('month', 'year', 'region_id'), temp_with_clim)
    },
    coords={
        'month': berkeley_months,
        'year': berkeley_years,
        'region_id': region_ids
    }
)

ds_out.to_netcdf('Berkeley_temperature_with_ERA5climatology_absolute_2025.nc')

print("完成，已将气候态加回异常值，生成绝对值温度数据。最后一年1、2月未处理。")


完成，已将气候态加回异常值，生成绝对值温度数据。最后一年1、2月未处理。


In [1]:
import xarray as xr

file_path = 'Berkeley_temperature_with_ERA5climatology_absolute_2025.nc'

try:
    ds = xr.open_dataset(file_path, engine='netcdf4')
    print(ds)
except Exception as e:
    print(f"读取文件时出错: {e}")

<xarray.Dataset> Size: 2MB
Dimensions:      (month: 12, year: 175, region_id: 237)
Coordinates:
  * month        (month) int64 96B 3 4 5 6 7 8 9 10 11 12 1 2
  * year         (year) int64 1kB 1850 1851 1852 1853 ... 2021 2022 2023 2024
  * region_id    (region_id) int64 2kB 0 1 2 3 4 5 6 ... 231 232 233 234 235 236
Data variables:
    temperature  (month, year, region_id) float32 2MB ...


In [11]:
import xarray as xr
import numpy as np

# 读取数据
ds_temp = xr.open_dataset('Berkeley_temperature_with_ERA5climatology_absolute_2025.nc')
ds_hottest = xr.open_dataset('region_hottest_month_1961_1990.nc')

# 变量提取
temp = ds_temp['temperature']         # (month, year, region_id)
hottest_month = ds_hottest['hottest_month']  # (region_id)

# 确认month坐标
month_coords = temp['month'].values   # 应该是 [3 4 5 6 7 8 9 10 11 12 1 2]

# 建立"月份到索引"映射
month_to_index = {m: idx for idx, m in enumerate(month_coords)}

# hottest_month的值是1-12，映射到对应索引
hottest_np = hottest_month.values  # shape (region_id,)
hottest_idx = np.array([month_to_index[m] for m in hottest_np])

# 提取温度数组
temp_np = temp.values  # (month, year, region_id)
years = temp['year'].values
n_year = len(years)
n_region = len(hottest_idx)

# 结果数组
result = np.empty((n_year, n_region), dtype=temp_np.dtype)

# 循环按区域提取最热月温度
for i_region in range(n_region):
    idx = hottest_idx[i_region]
    result[:, i_region] = temp_np[idx, :, i_region]

# 新数据集
ds_out = xr.Dataset(
    {
        't2m_hottest': (('year', 'region_id'), result)
    },
    coords={
        'year': years,
        'region_id': ds_temp['region_id'].values
    }
)

# 保存
ds_out.to_netcdf('Berkeley_hottest_month_by_region.nc')

print("完成，最热月温度已正确提取。")


完成，最热月温度已正确提取。


In [13]:
import xarray as xr

file_path = 'Berkeley_hottest_month_by_region_revised_by_ERA5.nc'

try:
    ds = xr.open_dataset(file_path, engine='netcdf4')
    print(ds)
except Exception as e:
    print(f"读取文件时出错: {e}")

<xarray.Dataset> Size: 169kB
Dimensions:      (year: 175, region_id: 237)
Coordinates:
  * year         (year) int64 1kB 1850 1851 1852 1853 ... 2021 2022 2023 2024
  * region_id    (region_id) int64 2kB 0 1 2 3 4 5 6 ... 231 232 233 234 235 236
Data variables:
    t2m_hottest  (year, region_id) float32 166kB ...


In [1]:
import xarray as xr
import numpy as np

# 读取数据
ds_temp = xr.open_dataset('Berkeley_temperature_with_ERA5climatology_absolute_2025.nc')
ds_hottest = xr.open_dataset('region_hottest_month_1961_1990.nc')

# 变量提取
temp = ds_temp['temperature']         # (month, year, region_id)
hottest_month = ds_hottest['hottest_month']  # (region_id,)

# 确认 month 坐标（应为 3~12, 1, 2）
month_coords = temp['month'].values
month_to_index = {m: idx for idx, m in enumerate(month_coords)}

# 区域数量、年份范围
hottest_np = hottest_month.values          # shape (region_id,)
hottest_idx = np.array([month_to_index[m] for m in hottest_np])  # 映射为 month 索引
years = temp['year'].values                # 全部年份
valid_years = years[(years >= 1900) & (years <= 2024)]  # 目标年份
n_year = len(valid_years)
n_region = len(hottest_idx)

# 温度数组 (month, year, region_id)
temp_np = temp.values
all_years = temp['year'].values
year_to_index = {y: i for i, y in enumerate(all_years)}

# 初始化结果数组
result = np.full((n_year, n_region), np.nan, dtype=temp_np.dtype)

# 遍历每个年份和区域
for i, year in enumerate(valid_years):
    for j in range(n_region):
        mon = hottest_np[j]  # 最热月：1-12
        if mon in [1, 2]:
            target_year = year - 1  # 向前一年
        else:
            target_year = year
        # 跳过无效年份（如 1899）
        if target_year not in year_to_index:
            continue
        year_idx = year_to_index[target_year]
        mon_idx = hottest_idx[j]
        result[i, j] = temp_np[mon_idx, year_idx, j]

# 构建新数据集
ds_out = xr.Dataset(
    {
        't2m_hottest': (('year', 'region_id'), result)
    },
    coords={
        'year': valid_years,
        'region_id': ds_temp['region_id'].values
    }
)

# 保存为 NetCDF
ds_out.to_netcdf('Berkeley_hottest_month_by_region_adjusted.nc')

print("完成：最热月温度提取完毕，含1-2月年份前移逻辑。")


完成：最热月温度提取完毕，含1-2月年份前移逻辑。


In [2]:
import xarray as xr

file_path = 'Berkeley_hottest_month_by_region_adjusted.nc'

try:
    ds = xr.open_dataset(file_path, engine='netcdf4')
    print(ds)
except Exception as e:
    print(f"读取文件时出错: {e}")

<xarray.Dataset> Size: 121kB
Dimensions:      (year: 125, region_id: 237)
Coordinates:
  * year         (year) int64 1kB 1900 1901 1902 1903 ... 2021 2022 2023 2024
  * region_id    (region_id) int64 2kB 0 1 2 3 4 5 6 ... 231 232 233 234 235 236
Data variables:
    t2m_hottest  (year, region_id) float32 118kB ...
